In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime
from functools import partial

# local imports
import sys
sys.path.append('../../../')
from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.Constants import CTE as CTE

from makedf.mcstat import get_MCstat_unc

from analysis_village.cc1pi.var_configs import *

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
save_result = False
save_fig = save_result

save_fig_base_dir = "/exp/sbnd/data/users/lpelegri/syts/all"
today_str = datetime.now().strftime("%Y%m%d")
save_fig_dir = path.join(save_fig_base_dir, "unfolding-fake_data_tests-{}".format(today_str))

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)

# Load MC dataframe

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')

#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
mc_bnb_df = load_df("/scratch/7DayLifetime/lpelegrina/cc1pi_5e18_CV.df", keys2load, 100)
mc_evt_df = mc_bnb_df['cc1pi']
mc_nu_df = mc_bnb_df['nudf']
mc_hdr_df = mc_bnb_df['hdr']

#Add weight column
data_tot_pot = 5.947e+18
mc_tot_pot = mc_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_evt_df))

#Do truth matchign
mc_evt_df = perform_truth_matching(mc_evt_df, mc_nu_df)
mc_nu_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_nu_df))

# Perform selection

In [ ]:
mc_obvious_cosmic_mask = mc_evt_df.slc.cut.obvious_cosmic
mc_t0_mask = mc_evt_df.slc.cut.t0
mc_is_inside_FV_mask = mc_evt_df.slc.cut.inside_FV
mc_nu_score_mask = mc_evt_df.slc.cut.nu_score
mc_track_mask = mc_evt_df.slc.cut.track
mc_shower_mask = mc_evt_df.slc.cut.shower 
mc_chi2_mask = mc_evt_df.slc.cut.MIP_candidates 
mc_angle_mask = mc_evt_df.slc.cut.angle 
mc_proton_BDT_mask = mc_evt_df.slc.cut.proton_BDT
mc_containment_mask = mc_evt_df.slc.cut.containment 
mc_michel_mask = mc_evt_df.slc.cut.michel 
mc_extra_pion_mask = mc_evt_df.slc.cut.extra_pion 
mc_energy_mask = mc_evt_df.slc.cut.energy

# 1. Define the order of cuts
mc_cut_sequence = [
    ("cosmic", mc_obvious_cosmic_mask),
    ("t0", mc_t0_mask),
    ("FV", mc_is_inside_FV_mask),
    ("nu_score", mc_nu_score_mask),
    ("track", mc_track_mask),
    ("chi2", mc_chi2_mask),
    ("shower", mc_shower_mask),
    ("angle", mc_angle_mask),
    ("proton_BDT", mc_proton_BDT_mask),
    ("containment", mc_containment_mask),
    ("michel", mc_michel_mask),
    ("extra_pion", mc_extra_pion_mask),
    ("energy", mc_energy_mask)
]

# 2. Build the cumulative masks
mc_cumulative_mak = None

for name, mask in mc_cut_sequence:
    if mc_cumulative_mak is None:
        mc_cumulative_mak = mask
    else:
        mc_cumulative_mak = mc_cumulative_mak & mask

In [ ]:
mc_evt_df = mc_evt_df[mc_cumulative_mak]

In [ ]:
mc_evt_df = (
        mc_evt_df
        .groupby(['__ntuple', 'entry', 'rec.slc..index'])
        .first()
    )
mc_evt_df = mc_evt_df.sort_index()

# Plot Syts

In [ ]:
var_configs = [
    VariableConfig.all_evts(), 
    VariableConfig.muon_momentum(),
    VariableConfig.muon_direction(),
    VariableConfig.pion_momentum(),
    VariableConfig.pion_direction(),
    VariableConfig.angle_between_candidates(),
    VariableConfig.num_protons(),
    VariableConfig.delta_pt(),
    VariableConfig.delta_alpha_T(),
    VariableConfig.delta_phi_T()
]

In [ ]:
# choose variable
var_config = VariableConfig.muon_momentum()

In [ ]:
file_dir = "/exp/sbnd/data/users/lpelegri/syst/frac_cov_matrices"

mcstat_syst = np.load(file_dir + "/mcstat_syst_dict.npz")
flux_syst = np.load(file_dir + "/flux_syst_dict.npz")
g4_syst = np.load(file_dir + "/g4_syst_dict.npz")
cosmics_syst = np.load(file_dir + "/cosmics_syst_dict.npz")
genie_syst = np.load(file_dir + "/genie_syst_dict_xsec.npz")

detvar_syst = np.load(file_dir + "/detvar_syst_dict.npz")

# flat uncertainties
pot_frac_unc = 0.02
ntargets_frac_unc = 0.01
nu_score= 0.03

# Unfolding

In [ ]:
# --- config for Wiener-SVD unfolding ---
C_type = 2
Norm_type = 0.5

# Closure test

In [ ]:
eps = 1e-8
ratio = True
approval = "internal"
textloc = [0.05, 0.55]
ax_ylim_ratio = 1.6
breakdown_type = "topology"

unfolding_plotter = partial(
    overlay_hists,
    breakdown_type=breakdown_type,
    mc_df=mc_evt_df,
    data_df=mc_evt_df,
    intime_df=None,
    ax_ylim_ratio=ax_ylim_ratio,
    ratio=ratio,
    textloc=textloc,
    approval=approval,
    save_fig=save_fig, 
)

In [ ]:
pot_str = get_pot_str(data_tot_pot)
plot_labels_hist = [var_config.var_labels[1], "Events / Bin (POT={})".format(pot_str), ""]

In [ ]:
ret = unfolding_plotter(var_config=var_config,
                    plot_labels=plot_labels_hist,
                    syst=None,
                    save_name=path.join(save_fig_dir, "{}_{}.png".format(var_config.var_save_name, breakdown_type)))

In [ ]:
ret = signal_hists(mc_evt_df, mc_nu_df, var_config, return_data=True, plot=False)

In [ ]:

print(var_config.bins)


save_fig_name = "{}/{}-reco_vs_true".format(save_fig_dir, var_config.var_save_name)
reco_vs_true, _, _ = np.histogram2d(ret["var_sel_truth"], 
                                    ret["var_sel_reco"], 
                                    weights=ret["wgt_sel_reco"], 
                                    bins=[var_config.bins, var_config.bins]) # Fix: wrap them!


evts_signal_truth, _, _ = plt.hist(ret["var_allmc"], bins=var_config.bins, weights=ret["wgt_allmc"], histtype="step", label="True Signal")
nevts_signal_sel_reco, _, _ = plt.hist(ret["var_sel_reco"], bins=var_config.bins, weights=ret["wgt_sel_reco"], histtype="step", label="Reco Selected Signal", color="k")
nevts_signal_sel_truth, _, _ = plt.hist(ret["var_sel_truth"], bins=var_config.bins, weights=ret["wgt_sel_truth"], histtype="step", label="True Selected Signal")
plt.legend()
plt.ylabel("Events")
plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[0])
plt.show()

plot_heatmap(reco_vs_true, 
             var_config.bins, 
             plot_labels=[var_config.var_labels[2], var_config.var_labels[1], "Smearing"],
             verbose=True,
             save_fig=save_fig, 
             save_name=save_fig_name)

In [ ]:
eff = ret["nevts_sel_truth"] / ret["nevts_allmc"]

save_fig_name = "{}/{}-response_matrix".format(save_fig_dir, var_config.var_save_name)
response = get_response_matrix(reco_vs_true, eff)

plot_heatmap(response, 
             var_config.bins, 
             plot_labels=[var_config.var_labels[2], var_config.var_labels[1], "Response"],
             save_fig=save_fig, 
             verbose=True,
             save_name=save_fig_name)

In [ ]:

measured = ret["nevts_sel_reco"]  
model    = ret["nevts_allmc"] 
Covariance = cov_from_fraccov(mcstat_syst[var_config.var_save_name], ret["nevts_sel_reco"])
unfold = WienerSVD(response, model, measured, Covariance, C_type, Norm_type)
'''
measured = ret["nevts_sel_reco"] * XSEC_UNIT 
model    = ret["nevts_allmc"] * XSEC_UNIT
Covariance = cov_from_fraccov(mcstat_syst[var_config.var_save_name], ret["nevts_sel_reco"]) * XSEC_UNIT**2
'''
unfold = WienerSVD(response, model, measured, Covariance, C_type, Norm_type)
print(unfold)

In [ ]:
import numpy as np
from scipy.stats import chi2 as chi2_dist

def get_chi2(data, model, cov, n_params=0):
    delta = model - data 
    inv_cov = np.linalg.inv(cov)

    chi2 = delta @ inv_cov @ delta
    ndof = len(data) - n_params
    pval = chi2_dist.sf(chi2, ndof)

    return chi2, pval

In [ ]:
def plot_unfolded_result(unfold, 
                         measured, 
                         models,
                         var_config, 
                         textloc=[0.05, 0.55],
                         approval="internal",
                         plot=True,
                         save_fig=False, 
                         save_name=None,
                         closure_test=True,
                         plot_xsec = True):

    bins = var_config.bins
    bin_centers = var_config.bin_centers
    bin_widths = np.diff(bins)
    
    if plot_xsec: 
        measured*XSEC_UNIT
        for midx, mkey in enumerate(models.keys()):
            models[mkey] = XSEC_UNIT*models[mkey]
        
    # unfolded result
    Unfolded = unfold['unfold']
    UnfoldedCov = unfold["UnfoldCov"]
    if plot_xsec:
        Unfolded = Unfolded*XSEC_UNIT
        UnfoldedCov = UnfoldedCov*XSEC_UNIT*XSEC_UNIT
        
    Unfolded_perwidth = Unfolded / bin_widths

    # --- stat uncertainties
    UnfoldCov_stat = unfold['StatUnfoldCov']
    if plot_xsec:
        UnfoldCov_stat = UnfoldCov_stat*XSEC_UNIT*XSEC_UNIT
    Unfold_uncert_stat = np.diag(UnfoldCov_stat)
    
    # --- syst uncertainties
    UnfoldCov_syst = unfold['SystUnfoldCov']
    if plot_xsec:
        UnfoldCov_syst = UnfoldCov_syst*XSEC_UNIT*XSEC_UNIT
    Unfold_uncert_syst = np.diag(UnfoldCov_syst)

    # --- decompose into norm and shape components
    # the first item in models dict is the nominal input model
    norm_model = list(models.keys())[0]
    SystUnfoldCov_norm, SystUnfoldCov_shape = Matrix_Decomp(models[norm_model], UnfoldCov_syst)
    Unfold_uncert_norm = np.sqrt(np.abs(np.diag(SystUnfoldCov_norm)))
    Unfold_uncert_shape = np.sqrt(np.abs(np.diag(SystUnfoldCov_shape)))

    # --- plot
    fig, ax = plt.subplots(figsize=(8.5, 7))
    # set err to 0 for closure test
    if closure_test:
        dummy_err = np.zeros_like(Unfolded_perwidth)
        bar_handle = plt.errorbar(bin_centers, Unfolded_perwidth, yerr=dummy_err, fmt='o', color='black')

    else:
        # plot shape uncertainty as error bars
        Unfold_uncert_stat_perwidth = Unfold_uncert_stat / bin_widths
        Unfold_uncert_shape_perwidth = Unfold_uncert_shape / bin_widths
        # Unfold_uncert_stat_shape_perwidth = Unfold_uncert_stat_perwidth + Unfold_uncert_shape_perwidth
        Unfold_uncert_stat_shape_perwidth = Unfold_uncert_shape_perwidth
        bar_handle = plt.errorbar(bin_centers, Unfolded_perwidth, yerr=Unfold_uncert_stat_shape_perwidth, fmt='o', color='black')

        # plot syst norm component as histogram at the bottom
        Unfold_uncert_norm_perwidth = Unfold_uncert_norm / bin_widths
        norm_handle = plt.bar(bin_centers, Unfold_uncert_norm_perwidth, width=bin_widths, label='Syst. error (norm)', alpha=0.5, color='gray')

    # divide measured & model by bin width
    measured_perwidth = measured / bin_widths
    reco_handle, = plt.step(bins, np.append(measured_perwidth, measured_perwidth[-1]), where='post', label='Meausred Signal (Input)')

    # --- get chi2 values for each model to compare
    chi2_vals = []
    p_values = []
    model_handles = []
    model_labels = []
    for midx, mkey in enumerate(models.keys()):
        model_smeared = unfold['AddSmear'] @ models[mkey]

        chi2_val, p_val = get_chi2(Unfolded, model_smeared, UnfoldCov_syst)
        chi2_vals.append(chi2_val)
        p_values.append(p_val)

        model_smeared_perwidth = model_smeared / bin_widths
        model_handle, = plt.step(bins, np.append(model_smeared_perwidth, model_smeared_perwidth[-1]), where='post')
        model_handles.append(model_handle)
        model_labels.append(f'$A_c \\otimes$ {mkey} ($\chi^2$ = {chi2_vals[midx]:.2f}/{len(bins)-1}), p-value = {p_values[midx]:.3f}')

    
    # legend
    if closure_test:
        handles = [bar_handle, reco_handle] + model_handles
        labels = ['Unfolded Asimov Data', 'Measured Signal'] + model_labels
    else:
        handles = [bar_handle, norm_handle, reco_handle] + model_handles
        labels = ['Unfolded', 'Norm. Syst. Unc.', 'Measured Signal'] + model_labels
    plt.legend(handles, labels, 
               loc='upper left', fontsize=12, frameon=False, ncol=1, bbox_to_anchor=(0.02, 0.98))


    plt.xlabel(var_config.var_labels[0])
    plt.ylabel(var_config.xsec_label)
    plt.xlim(bins[0], bins[-1])
    plt.ylim(0., np.max(Unfolded_perwidth)*1.7)

    # ==== plot additions
    textloc_x, textloc_ha = get_textloc_x(Unfolded_perwidth, var_config.bins, textloc)
    textloc_y = textloc[1]
    add_approval_text(approval, textloc_x, textloc_y, textloc_ha)

    if save_fig:
        plt.savefig(save_name+fig_ext, bbox_inches='tight', dpi=dpi)

    if plot == True:
        plt.show()
    else:
        plt.close()

In [ ]:
models = {"SBND Baseline model": model}
save_name = "{}-closure_test_output.pdf".format(var_config.var_save_name)
print(models)

plot_unfolded_result(unfold, 
                     measured, 
                     models, 
                     var_config,
                     save_fig=save_fig, 
                     save_name=save_name,
                     closure_test=True)

In [ ]:
save_fig_name = "{}/{}-{}-add_smear".format(save_fig_dir, var_config.var_save_name, "closure_test")
plot_heatmap(unfold["AddSmear"], 
             var_config.bins, 
             plot_labels=[var_config.var_labels[2], var_config.var_labels[1], "$A_c$"],
             save_fig=save_fig, 
             save_name=save_fig_name)

# Fake Data Test

In [ ]:

# test_name = "qe_test"
# scale_factor = 1.2
# weights_fake_data, weight_fakedata_signal_truth = fake_weight_obj.get_weights(test_name, scale_factor=scale_factor)

# test_name = "np_test"
# Np_scale = 2
# weights_fake_data, weight_fakedata_signal_truth = fake_weight_obj.get_weights(test_name, Np_scale=Np_scale)

# test_name = "sig_test"
# sig_scale = 1.2
# weights_fake_data, weight_fakedata_signal_truth = fake_weight_obj.get_weights(test_name, sig_scale=sig_scale)

# test_name = "q2_test_alpha_0.3"
# alpha = 0.3
# weights_fake_data, weight_fakedata_signal_truth = fake_weight_obj.get_weights(test_name, alpha=alpha)

# test_name = "costh_weight_scale_0.7"
# scale_factor = 0.7
# weights_fake_data, weight_fakedata_signal_truth = fake_weight_obj.get_weights(test_name, scale_factor=scale_factor)

# test_name = "proton_P_tilt_alpha_0.3"
# alpha = 0.3
# weights_fake_data, weight_fakedata_signal_truth = fake_weight_obj.get_weights(test_name, alpha=alpha)



In [ ]:
# use only stat unc and xsec unc for fake data tests
ret = signal_hists(mc_evt_df, mc_nu_df, var_config, return_data=True, plot=False)

covariance_frac = genie_syst[var_config.var_save_name] + mcstat_syst[var_config.var_save_name]
#covariance = cov_from_fraccov(covariance_frac, ret["nevts_sel_reco"]) * XSEC_UNIT**2
covariance = cov_from_fraccov(covariance_frac, ret["nevts_sel_reco"])

In [ ]:
unfolded_plot_labels = [var_config.var_labels[0], var_config.xsec_label]
smearmat_plot_labels = [var_config.var_labels[2], var_config.var_labels[1]]

In [ ]:
from analysis_village.cc1pi.systematics.fake_data_test_config import FakeDataWeights

fake_weight_obj = FakeDataWeights(mc_evt_df, mc_nu_df, var_config)

#test_name = "res_test"
#scale_factor = 1.2
#weights_fake_data, weight_fakedata_signal_truth = fake_weight_obj.get_weights(test_name, scale_factor=scale_factor)


test_name = "bump_0.6_0.0015_0.001"
bump_pos = 0.6
bump_width = 0.01
bump_height = 0.005
weights_fake_data, weight_fakedata_signal_truth = fake_weight_obj.get_weights(test_name, bump_pos=bump_pos, bump_width=bump_width, bump_height=bump_height)

print(len(mc_evt_df))
print(bump_height*len(mc_evt_df)/len(var_config.bin_centers))

In [ ]:
nevts_fakedata_reco, _ = np.histogram(ret["var_allsel_reco"], bins=var_config.bins, weights=weights_fake_data * ret["wgt_allsel_reco"])
nevts_nomdata_signal_truth, _ = np.histogram(ret["var_allmc"], bins=var_config.bins, weights= ret["wgt_allmc"])
nevts_fakedata_signal_truth, _ = np.histogram(ret["var_allmc"], bins=var_config.bins, weights=weight_fakedata_signal_truth*ret["wgt_allmc"])


fig, ax = plt.subplots()
plt.hist(var_config.bin_centers, var_config.bins, weights=nevts_fakedata_reco, histtype="step", label="all selected events")
plt.hist(var_config.bin_centers, var_config.bins, weights=nevts_nomdata_signal_truth, histtype="step", label="nominal MC, all signal")
plt.hist(var_config.bin_centers, var_config.bins, weights=nevts_fakedata_signal_truth, histtype="step", label="alternative MC, all signal")
plt.legend()
plt.show();

In [ ]:
fakedata_evt_df = mc_evt_df.copy()
fakedata_evt_df[('slc','wgt','','','','')] *= weights_fake_data

In [ ]:
syst = np.sqrt(np.diag(genie_syst[var_config.var_save_name]))

In [ ]:
print(syst)

In [ ]:
ret = overlay_hists(breakdown_type=breakdown_type,
                    mc_df=mc_evt_df,
                    data_df=fakedata_evt_df,
                    intime_df=None,
                    var_config=var_config,
                    plot_labels=plot_labels_hist,
                    ax_ylim_ratio=ax_ylim_ratio,
                    ratio=ratio,
                    syst=syst,
                    textloc=textloc,
                    approval=approval,
                    save_fig=save_fig, 
                    save_name=path.join(save_fig_dir, "{}_{}.png".format(var_config.var_save_name, breakdown_type)))

In [ ]:
print(ret["total_data"])

In [ ]:
print(ret["total_mc_bkgd"])

In [ ]:
#measured = (ret["total_data"] - ret["total_mc_bkgd"]) * XSEC_UNIT
#model = nevts_nomdata_signal_truth * XSEC_UNIT
measured = (ret["total_data"] - ret["total_mc_bkgd"])
model = nevts_nomdata_signal_truth 
unfold = WienerSVD(response, model, measured, covariance, C_type, Norm_type)

In [ ]:
#models = {"SBND Baseline Model": model, 
#          "Fake Data": nevts_fakedata_signal_truth * XSEC_UNIT}
models = {"SBND Baseline Model": model, 
          "Fake Data": nevts_fakedata_signal_truth}
save_name = "{}/{}-{}-unfolded_event_rates.pdf".format(save_fig_dir, test_name, var_config.var_save_name)
plot_unfolded_result(unfold, 
                     measured, 
                     models, 
                     var_config,
                     save_fig=save_fig, 
                     save_name=save_name,
                     closure_test=False)

In [ ]:
save_fig_name = "{}/{}-{}-add_smear".format(save_fig_dir, var_config.var_save_name, "closure_test")
plot_heatmap(unfold["AddSmear"], 
             var_config.bins, 
             plot_labels=[var_config.var_labels[2], var_config.var_labels[1], "$A_c$"],
             save_fig=save_fig, 
             save_name=save_fig_name)

In [ ]:
for column in mc_nu_df.truth.columns:
    print(column)